# Ordered Logistic Regression Results (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the Croissant dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Croissant Metadata instance, not a dictionary

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# List all record sets defined in the dataset schema
record_set_objs = dataset.record_sets  # List of Croissant RecordSet objects

if len(record_set_objs) == 0:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_set_objs:
        print(f"Record Set Name: {rs.name if hasattr(rs, 'name') else 'Unknown'}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - Name: {getattr(fld, 'name', 'Unknown')}, @id: {fld.id}")
        print("")

## 3. Data Extraction
Load data from each record set into pandas DataFrames, referencing everything by its `@id`.

If you do not see any record sets above, check the dataset documentation or use `dataset.schema` for a lower-level exploration. Here we continue by extracting all record sets, if any are available.

In [ ]:
# Prepare DataFrames for each record set (referenced by @id)
dataframes = {}
record_sets_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
    else:
        print(f"No data found for record set @id: {record_set_id}")

if len(dataframes) > 0:
    chosen_record_set_id = list(dataframes.keys())[0]  # Choose the first available
    print(f"\nColumns in record set @id: {chosen_record_set_id}")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print("No DataFrames available.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing using column `@id`s, such as filtering numeric values, normalizing, grouping, and handling missing values.

*If the record set is empty or has unknown columns, adjust the code or consult the schema for appropriate field `@id`s.*

In [ ]:
if len(dataframes) > 0:
    df = dataframes[chosen_record_set_id]
    # Try to infer a numeric field by inspecting types or column names
    numeric_field_id = None
    for col in df.columns:
        # Attempt to select a likely numeric column
        if df[col].dtype in ['float64', 'int64'] or 'likelihood' in col.lower() or 'coefficient' in col.lower():
            numeric_field_id = col
            break

    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (using @id as column):")
        display(filtered_df.head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() + 1e-8)
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to pick a categorical/grouping column (not the numeric_field)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and (
                pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])
            ) and df[col].nunique() < min(10, len(df)//2):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field} (using @id):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No suitable numeric field detected for EDA in this DataFrame.")
else:
    print("No DataFrame for EDA available.")

## 5. Visualization
Visualize data distributions or relationships using column `@id` references.

Below is an example histogram of a numeric field and a bar plot for a group-by summary (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field (by @id): {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Barplot of group-wise mean, if grouped_df is available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded a Croissant-described FAIR² dataset, reviewed record sets and their fields by `@id`, demonstrated data extraction, basic EDA, and visualized distributions—all referencing `@id` for clarity and reproducibility.

**Next steps** could include deeper statistical analysis, model-building, or joining with external data sources, all while maintaining reproducibility via entity `@id`s.